# v19 · Midline index: the 4-digit code T S DD over 96 characters, packed in base 96
`PtolemyDesktop/Archimedes/Maths/MidlineIndex.py` (git 2026-09-27; rules revised 2026-10-12). One character is a decimal `type*1000 + sign*100 + step`: type 0 Uppercase, 1 Lowercase, 2 Digit, 3 Punctuation (line feed and space lead it); sign is the side of the anchor `M`/`m` for letters, always 0 for digits, the half for punctuation (two halves of 17); step is the distance from the anchor (never more than 13), the digit itself, or the index in the half. 96 codes are valid; the whole text is a base-96 Horner number over each code's index in that sorted list.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, PROJ)
mi = load("PtolemyDesktop/Archimedes/Maths/MidlineIndex.py", "mi")
for ch in "AMZamz05\n !/:~":
    c = mi.encode_char(ch)
    print(repr(ch), "-> code %04d" % c, "->", repr(mi.decode_char(c)))
    assert mi.decode_char(c) == ch
addr = mi.encode_text("Hello, World -3!"); print(addr, mi.decode_text(addr, 16)); assert mi.verify("Hello, World -3!")

'A' -> code 0012 -> 'A'
'M' -> code 0000 -> 'M'
'Z' -> code 0113 -> 'Z'
'a' -> code 1012 -> 'a'
'm' -> code 1000 -> 'm'
'z' -> code 1113 -> 'z'
'0' -> code 2000 -> '0'
'5' -> code 2005 -> '5'
'\n' -> code 3000 -> '\n'
' ' -> code 3001 -> ' '
'!' -> code 3002 -> '!'
'/' -> code 3016 -> '/'
':' -> code 3100 -> ':'
'~' -> code 3116 -> '~'
2904025772685498955121040377056 Hello, World -3!


## The valid codes: exactly 96, nothing else decodes

In [3]:
from collections import Counter
print("VALID_CODES:", len(mi.VALID_CODES), "| K =", mi.K, "| by type:", dict(Counter(c // 1000 for c in mi.VALID_CODES)))
assert len(mi.VALID_CODES) == 96 == mi.K
assert sorted(mi.decode_char(c) for c in mi.VALID_CODES) == ["\n"] + [chr(x) for x in range(0x20, 0x7F)]
bad = 0
for code_ in range(10000):
    if code_ not in mi.CODE_INDEX:
        try: mi.decode_char(code_); raise AssertionError(code_)
        except ValueError: bad += 1
print("codes 0000..9999 rejected by decode_char:", bad, "(= 10000 - 96)")
print("digits: sign always 0 ->", [mi.encode_char(d) for d in "0123456789"])
print("line feed and space lead the punctuation codes:", mi.encode_char("\n"), mi.encode_char(" "), "| two halves of", mi.PUNCT_HALF, "characters:", repr(mi.PUNCT[:17]), repr(mi.PUNCT[17:]))

VALID_CODES: 96 | K = 96 | by type: {0: 26, 1: 26, 2: 10, 3: 34}
codes 0000..9999 rejected by decode_char: 9904 (= 10000 - 96)
digits: sign always 0 -> [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009]
line feed and space lead the punctuation codes: 3000 3001 | two halves of 17 characters: '\n !"#$%&\'()*+,-./' ':;<=>?@[\\]^_`{|}~'


## What it covers: case is exact, tab is four spaces, line feed is a code

In [4]:
print("case exact:", mi.decode_text(mi.encode_text("hello"), 5), mi.decode_text(mi.encode_text("HeLLo"), 5))
print("tab -> four spaces:", repr(mi.normalise("a\tb")))
for bad_ in ["a\rb", "caf\u00e9"]:
    try: mi.encode_text(bad_)
    except ValueError as e: print(repr(bad_), "->", e)
dom = "".join(c for c in mi.normalise(CORPUS) if 0x20 <= ord(c) <= 0x7E or c == "\n")
a = mi.encode_text(dom)
print("corpus (tabs to four spaces, line feeds kept):", len(dom), "chars | address bits:", a.bit_length(), "| bits/char:", round(a.bit_length()/len(dom), 3))
print("log2(96) =", round(math.log2(96), 3), "| log2(97) =", round(math.log2(97), 3), "| four decimal digits as written =", round(math.log2(10000), 3))
assert mi.decode_text(a, len(dom)) == dom

case exact: hello HeLLo
tab -> four spaces: 'a    b'
'a\rb' -> MidlineIndex covers the 96 supported characters (95 printable ASCII + line feed); got '\r'
'café' -> MidlineIndex covers the 96 supported characters (95 printable ASCII + line feed); got 'é'
corpus (tabs to four spaces, line feeds kept): 4000 chars | address bits: 26340 | bits/char: 6.585
log2(96) = 6.585 | log2(97) = 6.6 | four decimal digits as written = 13.288


The address is base 96, so it costs log₂ 96 = 6.585 bits per character: the minimum for 96 equally likely symbols, just under the 97-symbol Horner (6.60) because only tab is out of the domain. Storing the four decimal digits as written would cost 13.29 bits per character. Ordering the 95 symbols as T S DD makes the list a fixed permutation like any other base-96 `.perm`: it does not change the size, only which characters sit near each other.

In [5]:
print(mi.overhead_report())

{'midline_bits_per_char': 6.584962500721156, 'four_decimal_digits_bits_per_char': 13.287712379549449, 'horner97_bits_per_char': 6.599912842187128, 'true_minimum_bits_96symbols': 6.584962500721156, 'midline_cheaper_than_horner97': True, 'midline_overhead_vs_true_min': 1.0}
